# 🌕 Moon AI — Sovereign Option 1 25.7M Cloud GPU Preflight & Training Pipeline
### Chief ML Systems Architect: Prem Gaikwad (India 🇮🇳)
**Policy: OPTION 1 SOVEREIGN — Zero Teacher Influence — Pure Random Gaussian Scratch Initialization**

**What this notebook does:**
1. Inspects the physical NVIDIA Tesla T4 GPU hardware and runtime environment.
2. Clones the repository fresh, checks out the active feature branch (`feature/stage-2-3-multimodal-ui-sandbox-and-accelerated-training`), and verifies the exact commit.
3. Installs sovereign dependencies (`pyyaml`, `pydantic`, `fastapi`, `uvicorn`, `httpx`, `cryptography`, `tiktoken`).
4. Runs empirical hardware benchmarks (`scripts/validate_t4_empirical.py`).
5. Executes the 21-test Hardware Identity & Checkpoint Security test suite (`tests/test_stage_2_4_hardware_identity_and_checkpoint_security.py`).
6. Executes the 8-gate Production Preflight Engine (`scripts/preflight_pilot_training.py`).
7. Executes the 1-iteration Smoke Test with pre-training summary (`training/chat/train.py --config training/chat/phase_2_4_t4_pilot.yaml --smoke-test`).
8. Provides an explicitly guarded, authorization-locked cell for the Phase 2.5 100-step production pilot run.
9. Inspects the authentic configured checkpoint paths (`models/checkpoints/chat/moon_chat_phase_2_4_t4_pilot_v0.pt` and `best`).

> **IMPORTANT**: Runtime must be set to **T4 GPU**:
> Click **Runtime ➔ Change runtime type ➔ Select T4 GPU ➔ Save**.

In [ ]:
# Cell 1: Physical Hardware & GPU Identity Inspection
!nvidia-smi
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"

In [ ]:
# Cell 2: Clone Repository Fresh & Checkout Exact Audited Commit
!rm -rf NanoGPT
!git clone https://github.com/Prem759-0/NanoGPT.git
%cd NanoGPT

# Exact Audited Commit Pin (Detached HEAD Mode)
# Update this SHA when pinning to a newly audited and certified commit.
EXPECTED_COMMIT = "321e5a63d213b160415cc39038168008516fde5f"

import subprocess
subprocess.run(["git", "checkout", "--detach", EXPECTED_COMMIT], check=True)

actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == EXPECTED_COMMIT, (
    f"Commit verification failure: Expected {EXPECTED_COMMIT}, got {actual_commit}!"
)
print("✅ Exact audited commit verified:", actual_commit)
!git log -1 --oneline

In [ ]:
# Cell 3: Install Helper Dependencies (Zero External Pretrained Models)
!pip install -q pyyaml pydantic fastapi uvicorn httpx cryptography tiktoken

import sys
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"Compute Cap:     {torch.cuda.get_device_capability(0)}")
    print(f"Device Memory:   {torch.cuda.get_device_properties(0).total_memory / (1024**2):.1f} MB")

In [ ]:
# Cell 4: Empirical Hardware Identity & Tesla T4 Validation Gate
!python3 scripts/validate_t4_empirical.py

In [ ]:
# Cell 5: Run Stage 2.4 Hardware & Security Test Suite (21 Tests)
!python3 -m unittest tests.test_stage_2_4_hardware_identity_and_checkpoint_security -v

In [ ]:
# Cell 6: Run Phase 2.4.8 Production 8-Gate Preflight Engine
!python3 scripts/preflight_pilot_training.py

In [ ]:
# Cell 7: Dry-Run 1-Iteration Production Smoke Test (FP16 AMP, Scratch Init)
# Note: Smoke test saves an isolated, unpromotable checkpoint at:
# models/checkpoints/chat/smoke_test_ckpt.pt (cannot overwrite production checkpoints).
!python3 training/chat/train.py --config training/chat/phase_2_4_t4_pilot.yaml --smoke-test

## 🛑 Phase 2.5 Production Pilot (100 Iterations)
> **STOP CONDITION:** Do not execute the cell below without explicit authorization from Prem Gaikwad.
> Set `AUTHORIZED_BY_PREM = True` only after all preflights are certified and you personally intend to launch.

In [ ]:
# Cell 8: Phase 2.5 Live 100-Iteration Sovereign Pilot Training
AUTHORIZED_BY_PREM = False  # Set to True only with explicit launch authorization from Prem Gaikwad

if not AUTHORIZED_BY_PREM:
    print("🛑 HALTED: Phase 2.5 Production Pilot requires explicit launch authorization.")
    print("All 21 unit tests, 8 preflight gates, and 1-iteration smoke test must be certified first.")
    print("Do not modify this variable until you personally intend to launch.")
else:
    import subprocess
    launch_commit = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode("utf-8").strip()
    print(f"🚀 Launching Phase 2.5 100-step Sovereign Pilot Training at commit {launch_commit[:7]}...")
    !python3 training/chat/train.py --config training/chat/phase_2_4_t4_pilot.yaml

## 📊 Post-Training Verification & Checkpoint Integrity
Inspect the saved stage-isolated checkpoints after training completes.

In [ ]:
# Cell 9: Verify Checkpoint Integrity & Metadata
import os
import torch

ckpt_path = "models/checkpoints/chat/moon_chat_phase_2_4_t4_pilot_v0.pt"
best_ckpt_path = "models/checkpoints/chat/moon_chat_phase_2_4_t4_pilot_best.pt"
smoke_ckpt_path = "models/checkpoints/chat/smoke_test_ckpt.pt"

print("Checking Moon AI Chat checkpoint files:")
print(f"  • Final checkpoint exists:      {os.path.isfile(ckpt_path)} ({ckpt_path})")
print(f"  • Best checkpoint exists:       {os.path.isfile(best_ckpt_path)} ({best_ckpt_path})")
print(f"  • Smoke test checkpoint exists: {os.path.isfile(smoke_ckpt_path)} ({smoke_ckpt_path})")

target_to_inspect = ckpt_path if os.path.isfile(ckpt_path) else (smoke_ckpt_path if os.path.isfile(smoke_ckpt_path) else None)

if target_to_inspect:
    print(f"\nInspecting metadata from: {target_to_inspect}")
    ckpt = torch.load(target_to_inspect, map_location="cpu", weights_only=False)
    cfg = ckpt.get("config", {})
    print(f"  • Checkpoint status:     {cfg.get('status')}")
    print(f"  • Step / Iteration:      {ckpt.get('step', ckpt.get('iter_num'))}")
    print(f"  • Final validation loss: {ckpt.get('best_val_loss', ckpt.get('val_loss'))}")
    print(f"  • Model parameters:      {sum(p.numel() for p in ckpt['model'].values()):,}")
    print(f"  • Production eligible:   {cfg.get('is_production_eligible', False)}")
    print(f"  • Promotable:            {cfg.get('promotable', False)}")
    print(f"  • Initialization:        {cfg.get('initialization')}")